## Imports

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.express as px

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

## Configurações

In [9]:
PROCESSED_DIR = Path("../../data/interim")
RESULTS_DIR = Path("../../data/interim/deseq2")

FILTERED_EXPRESSION_PATH = PROCESSED_DIR / "microplastic_expression_filtered.csv"
METADATA_PATH = PROCESSED_DIR / "microplastic_metadata.csv"
WGCNA_INPUT_PATH = PROCESSED_DIR / "microplastic_vst_wgcna_input.csv"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

ALPHA = 0.05
LFC_THRESHOLD = 1.0  # apenas para sumarização/visualização

## Carregamento dos Dados Filtrados

In [3]:
expression_df = pd.read_csv(FILTERED_EXPRESSION_PATH)
metadata_df = pd.read_csv(METADATA_PATH)

print("Matriz filtrada:", expression_df.shape)
print("Metadados:", metadata_df.shape)

display(expression_df.head())
display(metadata_df.head())

Matriz filtrada: (12174, 25)
Metadados: (24, 9)


,gene_id,CTR_1,CTR_2,CTR_3,MA1_1,MA1_2,MA1_3,MA100_1,MA100_2,MA100_3,...,MB100_3,MC1_1,MC1_2,MC1_3,MC100_1,MC100_2,MC100_3,MD1_1,MD1_2,MD1_3
0,ENSG00000000003,357,327,329,276,226,355,336,285,323,...,334,279,319,370,310,358,398,276,365,336
1,ENSG00000000419,770,453,733,605,629,716,721,700,733,...,633,533,857,680,727,974,712,655,796,969
2,ENSG00000000457,27,115,31,26,33,35,30,31,7,...,44,32,17,27,41,68,67,42,38,47
3,ENSG00000000460,37,27,7,4,5,36,29,3,0,...,19,27,45,20,12,21,33,3,27,0
4,ENSG00000001036,980,943,970,807,886,1855,1094,780,1174,...,1046,1030,1175,1234,1286,1353,1381,1076,1243,1156


,particle_type,particle_size_um,particle_size_nm,concentration_gL,is_control,treatment_status,sample_id,group,replicate
0,control,NaN,NaN,0.0,True,control,CTR_1,CTR,1
1,control,NaN,NaN,0.0,True,control,CTR_2,CTR,2
2,control,NaN,NaN,0.0,True,control,CTR_3,CTR,3
3,polystyrene,1.0,1000.0,0.1,False,treated,MA1_1,MA1,1
4,polystyrene,1.0,1000.0,0.1,False,treated,MA1_2,MA1,2


## Preparação da Matriz para DEG (Differentially Expressed Genes)

In [4]:
# A matriz carregada tem genes como linhas e amostras como colunas, mas o DESeq2 espera o contrário.
# Transpondo a matriz para ter amostras como linhas e genes como colunas.

sample_ids = metadata_df["sample_id"].tolist()
expr_sample_cols = [c for c in expression_df.columns if c != "gene_id"]

missing_in_expr = sorted(set(sample_ids) - set(expr_sample_cols))
missing_in_meta = sorted(set(expr_sample_cols) - set(sample_ids))

if missing_in_expr or missing_in_meta:
    raise ValueError(
        f"Inconsistência entre matriz e metadata.\n"
        f"Ausentes na expressão: {missing_in_expr}\n"
        f"Ausentes no metadata: {missing_in_meta}"
    )

# Reordena colunas pela ordem do metadata
expression_df = expression_df[["gene_id"] + sample_ids]

# counts: samples x genes
counts = expression_df.set_index("gene_id").T
counts.index.name = "sample_id"

# Garante inteiros
counts = counts.astype(int)

# Metadata indexado por sample_id
metadata = metadata_df.set_index("sample_id").loc[counts.index].copy()

# Fator de condição principal para os contrastes
metadata["condition"] = metadata["group"].astype(str)

print("Counts:", counts.shape)
print("Metadata:", metadata.shape)

display(counts.iloc[:5, :5])
display(metadata.head())

Counts: (24, 12174)
Metadata: (24, 9)


gene_id,ENSG00000000003,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000001036
sample_id,,,,,
CTR_1,357,770,27,37,980
CTR_2,327,453,115,27,943
CTR_3,329,733,31,7,970
MA1_1,276,605,26,4,807
MA1_2,226,629,33,5,886


,particle_type,particle_size_um,particle_size_nm,concentration_gL,is_control,treatment_status,group,replicate,condition
sample_id,,,,,,,,,
CTR_1,control,NaN,NaN,0.0,True,control,CTR,1,CTR
CTR_2,control,NaN,NaN,0.0,True,control,CTR,2,CTR
CTR_3,control,NaN,NaN,0.0,True,control,CTR,3,CTR
MA1_1,polystyrene,1.0,1000.0,0.1,False,treated,MA1,1,MA1
MA1_2,polystyrene,1.0,1000.0,0.1,False,treated,MA1,2,MA1


In [5]:
# Validações finais antes do DESeq2

print("Condições disponíveis:")
print(sorted(metadata["condition"].unique()))

print("\nNúmero de amostras por condição:")
display(metadata["condition"].value_counts().sort_index())

print("\nAlgum valor ausente em counts?", counts.isna().sum().sum() > 0)
print("Algum valor negativo em counts?", (counts < 0).any().any())
print("Todos os counts são inteiros?", np.all(np.equal(np.mod(counts.to_numpy(), 1), 0)))

Condições disponíveis:
['CTR', 'MA1', 'MA100', 'MB1', 'MB100', 'MC1', 'MC100', 'MD1']

Número de amostras por condição:


condition
CTR      3
MA1      3
MA100    3
MB1      3
MB100    3
MC1      3
MC100    3
MD1      3
Name: count, dtype: int64


Algum valor ausente em counts? False
Algum valor negativo em counts? False
Todos os counts são inteiros? True


## Ajuste do Modelo DESeq2

In [6]:
dds = DeseqDataSet(
    counts=counts,
    metadata=metadata,
    design_factors="condition",
    quiet=False,
)

dds.deseq2()

print("Modelo DESeq2 ajustado com sucesso.")

Fitting size factors...
... done in 0.01 seconds.

/Users/daniel.hosomi/csvd-projects/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:281: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_.T + self.intercept_
/Users/daniel.hosomi/csvd-projects/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:281: RuntimeWarning: overflow encountered in matmul
  return X @ coef_.T + self.intercept_
/Users/daniel.hosomi/csvd-projects/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:281: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_.T + self.intercept_
Fitting dispersions...
... done in 0.72 seconds.

Fitting dispersion trend curve...
/Users/daniel.hosomi/csvd-projects/.venv/lib/python3.9/site-packages/pydeseq2/default_inference.py:211: RuntimeWarning: divide by zero encountered in matmul
  mu = covariates_fit @ coeffs
/Users/daniel.hosomi/csvd-projects/.venv/lib/python3.9/site-packages/pydeseq2/default_inference.

Modelo DESeq2 ajustado com sucesso.


... done in 0.59 seconds.

Calculating cook's distance...
... done in 0.01 seconds.

Replacing 0 outlier genes.



In [22]:
# Salva VST para cada Gene e amostra
contagens_normalizadas = dds.layers['normed_counts']
matriz_vst = np.log2(contagens_normalizadas + 1)

df_wgcna_input = pd.DataFrame(
    matriz_vst,
    index=dds.obs_names,
    columns=dds.var_names
)
df_wgcna_input = df_wgcna_input.T
df_wgcna_input.index.name = "gene_id"
df_wgcna_input = df_wgcna_input.reset_index()

# Remove a coluna 'sample_id' se existir
if 'sample_id' in df_wgcna_input.columns:
    df_wgcna_input = df_wgcna_input.drop(columns=['sample_id'])

print("Matriz VST pronta para o WGCNA!")
print("Dimensões:", df_wgcna_input.shape)

df_wgcna_input.to_csv(WGCNA_INPUT_PATH, index=False)

Matriz VST pronta para o WGCNA!
Dimensões: (12174, 25)


## Definição dos Contrastes

In [11]:
# Queremos contrastar as seguintes comparações, onde o primeiro elemento é o grupo de teste
# e o segundo é o grupo de referência (controle). O DESeq2 irá comparar o primeiro grupo contra
# o segundo, então a interpretação dos resultados será "diferença do grupo de teste em relação ao
# grupo de referência".

contrasts = [
    # Tratamento vs controle
    ("MA100", "CTR"),
    ("MB100", "CTR"),
    ("MC100", "CTR"),
    ("MA1", "CTR"),
    ("MB1", "CTR"),
    ("MC1", "CTR"),
    ("MD1", "CTR"),

    # Comparações pareadas por concentração entre 0.1 µm e 1 µm
    ("MA100", "MA1"),
    ("MB100", "MB1"),
    ("MC100", "MC1"),
]

print("Contrastes definidos:")
for tested, ref in contrasts:
    print(f"- {tested} vs {ref}")

Contrastes definidos:
- MA100 vs CTR
- MB100 vs CTR
- MC100 vs CTR
- MA1 vs CTR
- MB1 vs CTR
- MC1 vs CTR
- MD1 vs CTR
- MA100 vs MA1
- MB100 vs MB1
- MC100 vs MC1


## Execução dos Contrastes

In [12]:
## Função auxiliar para rodar contraste

def run_contrast(dds, tested_level: str, ref_level: str, alpha: float = 0.05) -> pd.DataFrame:
    """
    Executa um contraste DESeq2 e retorna um DataFrame com os resultados.
    """
    stat_res = DeseqStats(
        dds,
        contrast=["condition", tested_level, ref_level],
        alpha=alpha,
        quiet=False,
    )

    stat_res.summary()

    res = stat_res.results_df.copy().reset_index()
    res = res.rename(columns={"index": "gene_id"})
    res["contrast"] = f"{tested_level}_vs_{ref_level}"

    # Flags úteis
    res["significant"] = (~res["padj"].isna()) & (res["padj"] < alpha)
    res["direction"] = np.where(
        res["log2FoldChange"] > 0,
        f"up_in_{tested_level}",
        f"up_in_{ref_level}"
    )

    return res.sort_values(["padj", "pvalue"], na_position="last")

In [13]:
## Execução dos contrastes

all_results = {}
summary_rows = []

for tested, ref in contrasts:
    contrast_name = f"{tested}_vs_{ref}"
    print(f"\n=== Rodando contraste: {contrast_name} ===")

    res = run_contrast(dds, tested_level=tested, ref_level=ref, alpha=ALPHA)
    all_results[contrast_name] = res

    # Salva tabela individual
    output_file = RESULTS_DIR / f"deg_{contrast_name}.csv"
    res.to_csv(output_file, index=False)

    sig = res["significant"].fillna(False)
    up = sig & (res["log2FoldChange"] > 0)
    down = sig & (res["log2FoldChange"] < 0)

    summary_rows.append({
        "contrast": contrast_name,
        "n_genes_tested": res.shape[0],
        "n_significant": int(sig.sum()),
        "n_up_in_tested": int(up.sum()),
        "n_up_in_reference": int(down.sum()),
        "output_file": str(output_file),
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)


=== Rodando contraste: MA100_vs_CTR ===


Running Wald tests...
... done in 0.34 seconds.

Running Wald tests...


Log2 fold change & Wald test p-value: condition MA100 vs CTR
                    baseMean  log2FoldChange     lfcSE      stat    pvalue  \
gene_id                                                                      
ENSG00000000003   316.258491       -0.224380  0.188269 -1.191802  0.233339   
ENSG00000000419   698.215098        0.024341  0.190267  0.127932  0.898203   
ENSG00000000457    37.892108       -1.407073  0.586988 -2.397107  0.016525   
ENSG00000000460    19.577512       -1.252622  1.056639 -1.185477  0.235829   
ENSG00000001036  1067.507729       -0.067763  0.155100 -0.436900  0.662184   
...                      ...             ...       ...       ...       ...   
ENSG00000288920  7002.582703       -0.258460  0.141633 -1.824866  0.068021   
ENSG00000290292    60.538504        0.248909  0.518961  0.479630  0.631491   
ENSG00000291237   666.885913       -0.427451  0.196099 -2.179771  0.029274   
ENSG00000291317    19.591463        0.643881  0.881320  0.730588  0.465031   
ENS

... done in 0.26 seconds.

Running Wald tests...


Log2 fold change & Wald test p-value: condition MB100 vs CTR
                    baseMean  log2FoldChange     lfcSE      stat    pvalue  \
gene_id                                                                      
ENSG00000000003   316.258491       -0.172494  0.188535 -0.914919  0.360234   
ENSG00000000419   698.215098        0.063012  0.190404  0.330938  0.740692   
ENSG00000000457    37.892108       -0.676645  0.578257 -1.170145  0.241943   
ENSG00000000460    19.577512       -0.629313  1.047365 -0.600854  0.547938   
ENSG00000001036  1067.507729        0.028380  0.155130  0.182946  0.854841   
...                      ...             ...       ...       ...       ...   
ENSG00000288920  7002.582703       -0.572430  0.141744 -4.038481  0.000054   
ENSG00000290292    60.538504        0.353144  0.519035  0.680385  0.496261   
ENSG00000291237   666.885913       -0.034619  0.195540 -0.177041  0.859476   
ENSG00000291317    19.591463        0.992214  0.878134  1.129911  0.258514   
ENS

...

Log2 fold change & Wald test p-value: condition MA100 vs MA1
                    baseMean  log2FoldChange     lfcSE      stat    pvalue  \
gene_id                                                                      
ENSG00000000003   316.258491        0.069699  0.189383  0.368031  0.712850   
ENSG00000000419   698.215098        0.064899  0.190300  0.341033  0.733078   
ENSG00000000457    37.892108       -0.479328  0.595802 -0.804510  0.421103   
ENSG00000000460    19.577512       -0.398209  1.067587 -0.372999  0.709149   
ENSG00000001036  1067.507729       -0.257009  0.154782 -1.660455  0.096823   
...                      ...             ...       ...       ...       ...   
ENSG00000288920  7002.582703        0.358517  0.141781  2.528663  0.011450   
ENSG00000290292    60.538504        0.472550  0.520811  0.907336  0.364229   
ENSG00000291237   666.885913       -0.262974  0.196362 -1.339226  0.180497   
ENSG00000291317    19.591463       -0.153853  0.865503 -0.177762  0.858910   
ENS

... done in 0.26 seconds.

Running Wald tests...


Log2 fold change & Wald test p-value: condition MB100 vs MB1
                    baseMean  log2FoldChange     lfcSE      stat    pvalue  \
gene_id                                                                      
ENSG00000000003   316.258491       -0.109777  0.188859 -0.581261  0.561064   
ENSG00000000419   698.215098        0.037818  0.190403  0.198619  0.842561   
ENSG00000000457    37.892108        0.269452  0.588064  0.458203  0.646807   
ENSG00000000460    19.577512       -1.192346  1.042918 -1.143279  0.252923   
ENSG00000001036  1067.507729        0.046968  0.155200  0.302629  0.762173   
...                      ...             ...       ...       ...       ...   
ENSG00000288920  7002.582703       -0.153689  0.141850 -1.083463  0.278603   
ENSG00000290292    60.538504        0.325814  0.518992  0.627783  0.530146   
ENSG00000291237   666.885913        0.329024  0.196331  1.675860  0.093766   
ENSG00000291317    19.591463       -0.327940  0.856398 -0.382929  0.701773   
ENS

... done in 0.28 seconds.



,contrast,n_genes_tested,n_significant,n_up_in_tested,n_up_in_reference,output_file
0,MA100_vs_CTR,12174,1366,659,707,../../data/interim/deseq2/deg_MA100_vs_CTR.csv
1,MB100_vs_CTR,12174,1,1,0,../../data/interim/deseq2/deg_MB100_vs_CTR.csv
2,MC100_vs_CTR,12174,0,0,0,../../data/interim/deseq2/deg_MC100_vs_CTR.csv
3,MA1_vs_CTR,12174,239,107,132,../../data/interim/deseq2/deg_MA1_vs_CTR.csv
4,MB1_vs_CTR,12174,689,356,333,../../data/interim/deseq2/deg_MB1_vs_CTR.csv
5,MC1_vs_CTR,12174,1483,728,755,../../data/interim/deseq2/deg_MC1_vs_CTR.csv
6,MD1_vs_CTR,12174,1227,575,652,../../data/interim/deseq2/deg_MD1_vs_CTR.csv
7,MA100_vs_MA1,12174,147,91,56,../../data/interim/deseq2/deg_MA100_vs_MA1.csv
8,MB100_vs_MB1,12174,17,8,9,../../data/interim/deseq2/deg_MB100_vs_MB1.csv
9,MC100_vs_MC1,12174,1913,857,1056,../../data/interim/deseq2/deg_MC100_vs_MC1.csv


## Visualizações

In [14]:
def plot_volcano(
    results_df,
    title: str,
    alpha: float = 0.05,
    lfc_threshold: float = 1.0,
    max_points: int = 100000,
):
    df = results_df.copy()
    
    df = df.dropna(subset=["log2FoldChange", "pvalue"])

    if max_points is not None and df.shape[0] > max_points:
        df = df.sample(max_points, random_state=42)

    df["neglog10_padj"] = -np.log10(df["padj"].fillna(1.0).clip(lower=1e-300))

    df["Status"] = "não significativo"
    df.loc[(df["padj"] < alpha) & (df["log2FoldChange"] >= lfc_threshold), "Status"] = "up"
    df.loc[(df["padj"] < alpha) & (df["log2FoldChange"] <= -lfc_threshold), "Status"] = "down"

    fig = px.scatter(
        df,
        x="log2FoldChange",
        y="neglog10_padj",
        color="Status",
        color_discrete_map={
            "não significativo": "lightgray", 
            "up": "red", 
            "down": "blue"
        },
        hover_name=df.index,
        title=title,
        labels={
            "log2FoldChange": "log2 Fold Change", 
            "neglog10_padj": "-log10(padj)"
        }
    )

    fig.add_vline(x=lfc_threshold, line_dash="dash", line_color="black")
    fig.add_vline(x=-lfc_threshold, line_dash="dash", line_color="black")
    fig.add_hline(y=-np.log10(alpha), line_dash="dash", line_color="black")

    fig.update_traces(marker=dict(size=6, opacity=0.7))
    fig.update_layout(height=600, width=800)

    fig.show()

In [15]:
plot_volcano(
    all_results["MA100_vs_CTR"],
    title="Volcano Plot — MA100 vs CTR",
    alpha=ALPHA,
    lfc_threshold=LFC_THRESHOLD,
)

## Salvamento dos Resultados

In [16]:
all_results_df = pd.concat(all_results.values(), axis=0, ignore_index=True)

all_results_path = RESULTS_DIR / "deg_all_contrasts.csv"
all_results_df.to_csv(all_results_path, index=False)

print(f"Tabela consolidada salva em: {all_results_path}")
print("Dimensões:", all_results_df.shape)

Tabela consolidada salva em: ../../data/interim/deseq2/deg_all_contrasts.csv
Dimensões: (121740, 10)
